<h1>A simple tutorial for clustering</h1>

In this example you analyse a very simple dataset of countries, with their latitude and longitude as well as language information. For a more comprehensive dataset you can go to Kaggle (https://www.kaggle.com/) and download the countries dataset.

This example is a simple reinforcement of cluster analysis, to help form the basis of our future understanding of clusters.

<h1>Step 1: Import the libraries </h1>

In [ ]:
# cap the maths library to one thread — avoids the Windows/MKL
# memory-leak warning in K-Means (must run before sklearn is imported)
import os
os.environ["OMP_NUM_THREADS"] = "1"

# import the libraries we need:
# pandas      -> loading and working with the data table
# matplotlib  -> the scatter plots and elbow curve
# seaborn     -> not called directly, but importing it registers the 'mako'
#                colour scheme our cluster plots use
# KMeans      -> the clustering algorithm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans


<h1>Step 2: Import the data </h1>

In [ ]:
# read the dataset - 6 countries with their latitude, longitude and language
df = pd.read_csv('data/countries.csv')

# show the first 5 rows to check it loaded correctly
df.head()

<h1>Step 3: Scatter plot (Visualise the spread) </h1>

In [ ]:
# plot the raw data before clustering - each dot is one country
# longitude on x, latitude on y — so the plot reads like a world map
plt.scatter(df['Longitude'], df['Latitude'])
plt.xlabel('Longitude')
plt.ylabel('Latitude')



<h1>Step 4: Feature selection for clustering </h1>

In [ ]:
# X = columns 1 and 2 only (Latitude and Longitude) - the two numeric
# features K-Means will cluster on; the country names and language stay out
X = df.iloc[:,1:3]

# display the selected features
X

<h1>Step 5: Form the clusters </h1>

In [ ]:
# n_clusters=2  -> ask for two clusters (try 3 later)
# n_init=10     -> run K-Means 10 times from different random starts, keep the best
# random_state  -> fixed seed so everyone gets the same result
Kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)

In [ ]:
# fit = run the clustering: place 2 centre points, assign each country to its
# nearest centre, move each centre to the middle of its group, and repeat
# until nothing moves. With n_init=10 that whole process runs 10 times and
# the best result (lowest inertia) is kept.
Kmeans.fit(X)


<h1>Step 6: Cluster analysis on the clusters generated </h1>

In [ ]:
# fit_predict = fit the model, then return the cluster number for each row
# (strictly it re-fits — harmless here since random_state makes every fit identical)
clusters = Kmeans.fit_predict(X)

# one number per country, in the same order as the rows of the dataset
clusters

<h1>Step 7: Add the Cluster columun to the data set</h1>

In [ ]:
# work on a copy so the original dataset stays untouched
clustered_data = df.copy()

# add the cluster number for each country as a new column
clustered_data['Cluster'] = clusters

# show the full table — countries with the same Cluster number were grouped together
clustered_data

<h1>Step 8: Visualise the Clusters </h1>

In [ ]:
# plot the countries again, but colour each point by its cluster number
# c=      -> the value that decides each point's colour (the Cluster column)
# cmap    -> the colour scheme that turns those numbers into colours
#            ('mako' comes from seaborn — that's why we import it)
plt.scatter(clustered_data['Longitude'], clustered_data['Latitude'], c=clustered_data['Cluster'], cmap='mako')
plt.xlabel('Longitude')
plt.ylabel('Latitude')


<h1>Step 9: Determine the optimum number of clusters using the Elbow Method </h1>
One of the most challenging parts to answer is how many optimum clusters are required of a dataset. In this example, it is quite simple because we only have two features: Latitude and Longitude. In the real-world scenarios, you can have more features and it may not be that clear on how the clusters are being formed. 
So, lets have a look into one of the most popular methods that determines the optimum number of clusters, that is the Elbow method.

In [ ]:
# Elbow method to determine the optimum number of clusters
# fit K-Means for k = 1 to 5 and record the inertia (sum of squared
# distances from each point to its cluster centre) for each k
inertia = []                                                    # collect one inertia value per k

for i in range(1, 6):                                           # i = 1, 2, 3, 4, 5
    kmeans = KMeans(n_clusters=i, n_init=10, random_state=42)   # same settings as before, but k = i
    kmeans.fit(X)                                               # run the clustering for this k
    inertia.append(kmeans.inertia_)                             # inertia_ = total spread within the clusters (lower = tighter)

# plot k against inertia — the "elbow" is where the line stops dropping steeply
plt.figure(figsize=(12,6))
plt.plot(range(1, 6), inertia, marker='o')                      # marker='o' puts a dot at each k
plt.title('Elbow Curve')
plt.xlabel('Number of Clusters')
plt.ylabel('Squared Sum (Inertia)')

Looking at the above graph, looks like 3 is a better number for clusters

In [ ]:
# rebuild the model with the elbow's answer: 3 clusters
# (same settings as before — 10 random starts, fixed seed)
Kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
Kmeans.fit(X)

In [ ]:
# get the cluster number for each country, this time with 3 clusters
clusters = Kmeans.fit_predict(X)
clusters

In [ ]:
# fresh copy of the dataset with the new 3-cluster labels attached
clustered_data = df.copy()
clustered_data['Cluster'] = clusters
clustered_data

In [ ]:
# final picture: the three clusters on the map-style layout,
# coloured by the new Cluster column — one colour per continent group
plt.scatter(clustered_data['Longitude'], clustered_data['Latitude'], c=clustered_data['Cluster'], cmap='mako')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

<h1>What did we learn?</h1>

- **K-Means groups by distance.** With no labels to learn from, it split the countries purely on how close their coordinates are — and the geography fell out naturally: Asia-Pacific, North America, Europe.
- **Cluster numbers are arbitrary.** Run to run, "cluster 0" can become "cluster 2". The grouping is the result, not the numbers.
- **K-Means depends on its random start.** It can land in a poor solution, which is why we run it multiple times (`n_init=10`) and keep the best, and fix `random_state` so results are repeatable.
- **The elbow method chooses k.** Inertia always falls as k grows — the elbow is where it stops falling *fast*. Here the drops were 49362 → 14166 → 3011, then almost nothing: the elbow is at 3.
- **You must ask for k.** K-Means never decides how many clusters exist — that judgement stayed with us.